In [30]:
#fish image layerstack

import cv2
import numpy as np
import os
import rasterio
from rasterio.transform import from_origin
from sklearn.metrics import mean_squared_error

def sift_keypoints_descriptors(img, nfeatures=0, nOctaveLayers=3, contrastThreshold=0.04, edgeThreshold=10, sigma=1.6):
    sift = cv2.SIFT_create(
        nfeatures=nfeatures,
        nOctaveLayers=nOctaveLayers,
        contrastThreshold=contrastThreshold,
        edgeThreshold=edgeThreshold,
        sigma=sigma
    )
    keypoints, descriptors = sift.detectAndCompute(img, None)
    return keypoints, descriptors

def match_features(des1, des2, max_matches=500):
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)
    return matches[:max_matches]

def apply_transformation(kp1, kp2, matches, img, ref_img):
    src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    matrix, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
    aligned_img = cv2.warpPerspective(img, matrix, (ref_img.shape[1], ref_img.shape[0]))
    return aligned_img

def save_as_rasterio_tiff(blue, green, red, red_edge, nir, output_path):
    height, width = blue.shape
    transform = from_origin(0, 0, 1, 1)

    with rasterio.open(
        output_path, 'w',
        driver='GTiff',
        height=height,
        width=width,
        count=5,
        dtype=blue.dtype,
        crs='+proj=latlong',
        transform=transform
    ) as dst:
        dst.write(blue, 1)
        dst.write(green, 2)
        dst.write(red, 3)
        dst.write(red_edge, 4)
        dst.write(nir, 5)

import os

def save_layerstacked_images(aligned_bands, output_directory, base_name):
    """
    Save aligned bands as a single TIFF file.
    :param aligned_bands: Dictionary containing aligned bands.
    :param output_directory: Directory to save the layerstacked TIFF files.
    :param base_name: Base name for the output file.
    """
    output_path = os.path.join(output_directory, f"{base_name}_layerstacked.tif")
    save_as_rasterio_tiff(
        aligned_bands["blue"],
        aligned_bands["green"],
        aligned_bands["red"],
        aligned_bands["red_edge"],
        aligned_bands["nir"],
        output_path,
    )

def process_folder(folder_path, output_directory):
    """
    Process all files in a folder for layerstacking.
    :param folder_path: Path to the folder containing input images.
    :param output_directory: Path to save layerstacked images.
    """
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    
    files = sorted(os.listdir(folder_path))
    grouped_images = {}

    # Group images by the prefix (e.g., IMG_XXXX)
    for file in files:
        if not file.endswith(".tif"):
            continue
        
        prefix = "_".join(file.split("_")[:2])  # Group by IMG_XXXX
        if prefix not in grouped_images:
            grouped_images[prefix] = {}
        
        if "_1" in file:
            grouped_images[prefix]["blue"] = os.path.join(folder_path, file)
        elif "_2" in file:
            grouped_images[prefix]["green"] = os.path.join(folder_path, file)
        elif "_3" in file:
            grouped_images[prefix]["red"] = os.path.join(folder_path, file)
        elif "_4" in file:
            grouped_images[prefix]["nir"] = os.path.join(folder_path, file)
        elif "_5" in file:
            grouped_images[prefix]["red_edge"] = os.path.join(folder_path, file)
    
    # Process each group
    for base_name, band_paths in grouped_images.items():
        print(f"Processing group: {base_name}")
        
        # Ensure all bands exist
        if not all(band in band_paths for band in ["blue", "green", "red", "nir", "red_edge"]):
            print(f"Skipping {base_name}: Missing one or more bands.")
            continue
        
        # Read images
        bands = {name: cv2.imread(path, cv2.IMREAD_GRAYSCALE) for name, path in band_paths.items()}
        
        # Align bands to the RedEdge
        ref_img = bands["red_edge"]
        aligned_bands = {}
        
        for band_name, img in bands.items():
            if band_name == "red_edge":
                aligned_bands[band_name] = img  # Reference band doesn't need alignment
                continue
            
            kp1, des1 = sift_keypoints_descriptors(img)
            kp2, des2 = sift_keypoints_descriptors(ref_img)
            matches = match_features(des1, des2, max_matches=1500)
            aligned_bands[band_name] = apply_transformation(kp1, kp2, matches, img, ref_img)
        
        # Save the layerstacked image
        save_layerstacked_images(aligned_bands, output_directory, base_name)
        print(f"Saved layerstacked image: {base_name}_layerstacked.tif")


# Specify the folder containing the images and the output path
folder_path = "F:\\Re-MX_data_24to28.07.2024\\Corrected_27.07.2024\\Corrected_12_Hours_sample\\0001SET\\000"
output_path = "F:\\Re-MX_data_24to28.07.2024\\Corrected_27.07.2024\\Corrected_12_Hours_sample\\0001SET\\layerstack"
#output_path = "C:\\Tarun\\FISH\\layerstack"
process_folder(folder_path, output_path)

Processing group: IMG_0001
Saved layerstacked image: IMG_0001_layerstacked.tif
Processing group: IMG_0002
Saved layerstacked image: IMG_0002_layerstacked.tif
Processing group: IMG_0005
Saved layerstacked image: IMG_0005_layerstacked.tif


In [1]:
# MXIndices 

import rasterio
from rasterio.plot import show
import numpy as np
import spyndex
import os

# Define parameters for index calculation
params_template = {
    'C1': 6.0,
    'C2': 7.5,
    'L': 1.0,
    'alpha': 0.1,
    'beta': 0.05,
    'c': 1.0,
    'cexp': 1.16,
    'g': 2.5,
    'gamma': 1.0,
    'nexp': 2.0,
    'omega': 2.0,
    'p': 2.0,
    'sigma': 0.5,
    'sla': 1.0,
    'slb': 0.5
}

# List of indices to calculate
index_list = ['ARI', 'CG', 'CIRE', 'CVI', 'CCCI', 'DVI', 'EVI', 'EVI2',
              'ExG', 'ENDVI', 'FCVI', 'GNDVI', 'GRVI', 'GCI', 'GSAVI',
              'IKAW', 'MNLI', 'MCARI', 'MSRI', 'MCARI2', 'MSAVI',
              'mND705', 'mSR', 'MTCI', 'NDVI', 'NLI', 'NDREI', 'NGRDI', 'NDCI',
              'CIG', 'NRI', 'OSAVI', 'RDVI', 'RVI', 'RECI', 'RGRI', 'REIP',
              'RENDVI', 'SAVI', 'GM1', 'TCARI', 'TVI', 'WDRVI']

def calculate_and_save_index(index_name, params, profile, output_dir):
    try:
        # Calculate the index
        index_result = spyndex.computeIndex(index=index_name, **params)
        
        # Save the index as a GeoTIFF file
        index_output_path = os.path.join(output_dir, f'{index_name.lower()}.tif')
        profile.update(dtype=rasterio.float32, count=1)
        
        with rasterio.open(index_output_path, 'w', **profile) as dst:
            dst.write(index_result.astype(rasterio.float32), 1)
        
        print(f'{index_name} saved to {index_output_path}')
    except Exception as e:
        print(f'Error calculating {index_name}: {e}')

def process_layerstacked_images(input_folder, output_root):
    # Ensure the output directory exists
    os.makedirs(output_root, exist_ok=True)

    for file in os.listdir(input_folder):
        if file.endswith('.tif'):
            # Define paths
            image_path = os.path.join(input_folder, file)
            layerstack_name = os.path.splitext(file)[0]
            output_dir = os.path.join(output_root, layerstack_name)
            os.makedirs(output_dir, exist_ok=True)

            print(f'Processing {image_path}...')

            # Open the Micasense image
            with rasterio.open(image_path) as src:
                # Read the image bands into numpy arrays
                red = src.read(1)
                green = src.read(2)
                blue = src.read(3)
                nir = src.read(4)
                red_edge = src.read(5)

                # Create a profile for the output files
                profile = src.profile

            # Set the parameters for index calculation
            params = params_template.copy()
            params.update({'N': nir, 'N1': nir, 'N2': nir,
                           'R': red, 'RE1': red_edge, 'RE2': red_edge, 
                           'RE3': red_edge, 'B': blue, 'G': green})
            
            # Calculate and save each index
            for index_name in index_list:
                calculate_and_save_index(index_name, params, profile, output_dir)
            
            print(f'Finished processing {layerstack_name}.\n')

# Example usage
input_folder = r"F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack"  # Folder containing layerstacked images
output_root = r"F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices"        # Root folder for indices
process_layerstacked_images(input_folder, output_root)


Processing F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\IMG_0001_layerstacked.tif...
ARI saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices\IMG_0001_layerstacked\ari.tif
Error calculating CG: CG is not a valid Spectral Index!
CIRE saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices\IMG_0001_layerstacked\cire.tif


<string>:1: RuntimeWarning: divide by zero encountered in divide
<string>:1: RuntimeWarning: invalid value encountered in subtract
C:\Users\BDL\anaconda3\envs\micasense\lib\site-packages\rasterio\__init__.py:343: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(
<string>:1: RuntimeWarning: divide by zero encountered in divide
C:\Users\BDL\anaconda3\envs\micasense\lib\site-packages\rasterio\__init__.py:343: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(
<string>:1: RuntimeWarning: divide by zero encountered in divide
<string>:1: RuntimeWarning: invalid value encountered in divide
C:\Users\BDL\anaconda3\envs\mic

CVI saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices\IMG_0001_layerstacked\cvi.tif
Error calculating CCCI: CCCI is not a valid Spectral Index!
DVI saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices\IMG_0001_layerstacked\dvi.tif
EVI saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices\IMG_0001_layerstacked\evi.tif


C:\Users\BDL\anaconda3\envs\micasense\lib\site-packages\rasterio\__init__.py:343: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(
<string>:1: RuntimeWarning: divide by zero encountered in divide
<string>:1: RuntimeWarning: invalid value encountered in divide
C:\Users\BDL\anaconda3\envs\micasense\lib\site-packages\rasterio\__init__.py:343: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(
C:\Users\BDL\anaconda3\envs\micasense\lib\site-packages\rasterio\__init__.py:343: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotran

EVI2 saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices\IMG_0001_layerstacked\evi2.tif
ExG saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices\IMG_0001_layerstacked\exg.tif
Error calculating ENDVI: ENDVI is not a valid Spectral Index!
FCVI saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices\IMG_0001_layerstacked\fcvi.tif


...

C:\Users\BDL\anaconda3\envs\micasense\lib\site-packages\rasterio\__init__.py:343: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(
<string>:1: RuntimeWarning: divide by zero encountered in divide
<string>:1: RuntimeWarning: invalid value encountered in divide
C:\Users\BDL\anaconda3\envs\micasense\lib\site-packages\rasterio\__init__.py:343: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(


SAVI saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices\IMG_0095_layerstacked\savi.tif
GM1 saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices\IMG_0095_layerstacked\gm1.tif


<string>:1: RuntimeWarning: divide by zero encountered in divide
<string>:1: RuntimeWarning: invalid value encountered in divide
<string>:1: RuntimeWarning: invalid value encountered in multiply
C:\Users\BDL\anaconda3\envs\micasense\lib\site-packages\rasterio\__init__.py:343: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(


TCARI saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices\IMG_0095_layerstacked\tcari.tif
TVI saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices\IMG_0095_layerstacked\tvi.tif
WDRVI saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices\IMG_0095_layerstacked\wdrvi.tif
Finished processing IMG_0095_layerstacked.



<string>:1: RuntimeWarning: divide by zero encountered in divide
<string>:1: RuntimeWarning: invalid value encountered in divide
C:\Users\BDL\anaconda3\envs\micasense\lib\site-packages\rasterio\__init__.py:343: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(
C:\Users\BDL\anaconda3\envs\micasense\lib\site-packages\rasterio\__init__.py:343: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(


In [5]:
# MXIndices 
#error handling
import numpy as np
import rasterio
import os
import spyndex
from rasterio.errors import NotGeoreferencedWarning
import warnings

# Suppress NotGeoreferencedWarning
warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)

# Function to safely compute indices with masking
def safe_compute_index(index_name, params):
    try:
        # Compute the index using spyndex
        with np.errstate(divide='ignore', invalid='ignore'):
            index_result = spyndex.computeIndex(index=index_name, **params)
            # Mask invalid values (divide by zero, NaN, infinity)
            index_result = np.where(np.isfinite(index_result), index_result, 0.0)
        return index_result
    except Exception as e:
        print(f"Error calculating {index_name}: {e}")
        return None

# Function to calculate and save an index
def calculate_and_save_index(index_name, params, profile, output_dir):
    index_result = safe_compute_index(index_name, params)
    if index_result is not None:
        index_output_path = os.path.join(output_dir, f"{index_name.lower()}.tif")
        profile.update(dtype=rasterio.float32, count=1)

        with rasterio.open(index_output_path, "w", **profile) as dst:
            dst.write(index_result.astype(rasterio.float32), 1)

        print(f"{index_name} saved to {index_output_path}")

# Function to process all layerstacked images in a folder
def process_layerstacked_images(input_folder, output_root):
    os.makedirs(output_root, exist_ok=True)

    for file in os.listdir(input_folder):
        if file.endswith(".tif"):
            image_path = os.path.join(input_folder, file)
            layerstack_name = os.path.splitext(file)[0]
            output_dir = os.path.join(output_root, layerstack_name)
            os.makedirs(output_dir, exist_ok=True)

            print(f"Processing {image_path}...")

            # Open the image
            with rasterio.open(image_path) as src:
                red = src.read(1)
                green = src.read(2)
                blue = src.read(3)
                nir = src.read(4)
                red_edge = src.read(5)
                profile = src.profile

            # Prepare parameters for index calculation
            params = {
                "N": nir,
                "N1": nir,
                "N2": nir,
                "R": red,
                "RE1": red_edge,
                "RE2": red_edge,
                "RE3": red_edge,
                "B": blue,
                "G": green,
                "C1": 6.0,
                "C2": 7.5,
                "L": 1.0,
                "alpha": 0.1,
                "beta": 0.05,
                "c": 1.0,
                "cexp": 1.16,
                "g": 2.5,
                "gamma": 1.0,
                "nexp": 2.0,
                "omega": 2.0,
                "p": 2.0,
                "sigma": 0.5,
                "sla": 1.0,
                "slb": 0.5,
            }

            # Indices to calculate
            #index_list = ["NDVI", "EVI", "SAVI", "GNDVI", "GRVI", "CIRE", "DVI", "MCARI", "RVI"]
            index_list = ['ARI', 'CG', 'CIRE', 'CVI', 'CCCI', 'DVI', 'EVI', 'EVI2',
              'ExG', 'ENDVI', 'FCVI', 'GNDVI', 'GRVI', 'GCI', 'GSAVI',
              'IKAW', 'MNLI', 'MCARI', 'MSRI', 'MCARI2', 'MSAVI',
              'mND705', 'mSR', 'MTCI', 'NDVI', 'NLI', 'NDREI', 'NGRDI', 'NDCI',
              'CIG', 'NRI', 'OSAVI', 'RDVI', 'RVI', 'RECI', 'RGRI', 'REIP',
              'RENDVI', 'SAVI', 'GM1', 'TCARI', 'TVI', 'WDRVI']

            for index_name in index_list:
                calculate_and_save_index(index_name, params, profile, output_dir)

            print(f"Finished processing {layerstack_name}.\n")

# Example usage
input_folder = r"F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack"  # Folder containing layerstacked images
output_root = r"F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices2"        # Root folder for indices
process_layerstacked_images(input_folder, output_root)

Processing F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\IMG_0001_layerstacked.tif...
ARI saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices2\IMG_0001_layerstacked\ari.tif
Error calculating CG: CG is not a valid Spectral Index!
CIRE saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices2\IMG_0001_layerstacked\cire.tif
CVI saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices2\IMG_0001_layerstacked\cvi.tif
Error calculating CCCI: CCCI is not a valid Spectral Index!
DVI saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices2\IMG_0001_layerstacked\dvi.tif
EVI saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices2\IMG_0001_layerstacked\evi.tif
EVI2 saved to F:\Re-MX_data_24to28.07.2024\Corrected_24.07.2024\Corrected_4 Hours\layerstack\indices2\IMG_00

KeyboardInterrupt: 

In [27]:
#Normalized MXIndices
#error handling
import numpy as np
import rasterio
import os
import spyndex
from rasterio.errors import NotGeoreferencedWarning
import warnings

# Suppress NotGeoreferencedWarning
warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)

def normalize_band(band, scale_factor=10000):
    """
    Normalize reflectance values to the range [0, 1].
    """
    normalized_band = band.astype(np.float32) / scale_factor
    normalized_band = np.clip(normalized_band, 0, 1)  # Ensure values are within range
    return normalized_band

def safe_compute_index(index_name, params):
    """
    Compute index safely, handling invalid and extreme values.
    """
    try:
        with np.errstate(divide='ignore', invalid='ignore'):
            index_result = spyndex.computeIndex(index=index_name, **params)
            # Replace invalid or extreme values with NaN
            index_result = np.where(np.isfinite(index_result), index_result, np.nan)
        return index_result
    except Exception as e:
        print(f"Error calculating {index_name}: {e}")
        return None

def calculate_and_save_index(index_name, params, profile, output_dir):
    """
    Calculate and save an index to a GeoTIFF.
    """
    index_result = safe_compute_index(index_name, params)
    if index_result is not None:
        index_output_path = os.path.join(output_dir, f"{index_name.lower()}.tif")
        profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

        with rasterio.open(index_output_path, "w", **profile) as dst:
            dst.write(index_result.astype(rasterio.float32), 1)

        print(f"{index_name} saved to {index_output_path}")

def process_layerstacked_images(input_folder, output_root):
    """
    Process all layerstacked images in a folder and calculate indices.
    """
    os.makedirs(output_root, exist_ok=True)

    for file in os.listdir(input_folder):
        if file.endswith(".tif"):
            image_path = os.path.join(input_folder, file)
            layerstack_name = os.path.splitext(file)[0]
            output_dir = os.path.join(output_root, layerstack_name)
            os.makedirs(output_dir, exist_ok=True)

            print(f"Processing {image_path}...")

            # Open the image
            with rasterio.open(image_path) as src:
                red = normalize_band(src.read(1))
                green = normalize_band(src.read(2))
                blue = normalize_band(src.read(3))
                nir = normalize_band(src.read(4))
                red_edge = normalize_band(src.read(5))
                profile = src.profile

            # Mask invalid pixels
            valid_mask = (red > 0) & (green > 0) & (blue > 0) & (nir > 0) & (red_edge > 0)
            red[~valid_mask] = 0
            green[~valid_mask] = 0
            blue[~valid_mask] = 0
            nir[~valid_mask] = 0
            red_edge[~valid_mask] = 0

            # Prepare parameters for index calculation
            params = {
                "N": nir,
                "N1": nir,
                "N2": nir,
                "R": red,
                "RE1": red_edge,
                "RE2": red_edge,
                "RE3": red_edge,
                "B": blue,
                "G": green,
                "C1": 6.0,
                "C2": 7.5,
                "L": 1.0,
                "alpha": 0.1,
                "beta": 0.05,
                "c": 1.0,
                "cexp": 1.16,
                "g": 2.5,
                "gamma": 1.0,
                "nexp": 2.0,
                "omega": 2.0,
                "p": 2.0,
                "sigma": 0.5,
                "sla": 1.0,
                "slb": 0.5,
            }

            # Indices to calculate
            #index_list = ["NDVI", "EVI", "SAVI", "GNDVI", "GRVI", "CIRE", "DVI", "MCARI", "RVI"]
            index_list = ['ARI', 'CIRE', 'CVI', 'DVI', 'EVI', 'EVI2','ExG', 'FCVI', 'GNDVI', 'GRVI', 'GSAVI',
              'IKAW', 'MNLI', 'MCARI', 'MCARI2', 'MSAVI','MTCI', 'NDVI', 'NLI', 'NDREI', 'NGRDI', 'NDCI',
              'CIG', 'OSAVI', 'RDVI', 'RVI', 'RGRI', 'RENDVI', 'SAVI', 'GM1', 'TCARI', 'TVI', 'WDRVI']

            for index_name in index_list:
                calculate_and_save_index(index_name, params, profile, output_dir)

            print(f"Finished processing {layerstack_name}.\n")

# Example usage
input_folder = r"F:\Re-MX_data_24to28.07.2024\Corrected_27.07.2024\Corrected_12_Hours_sample\0001SET\layerstack"  # Folder containing layerstacked images
output_root = r"F:\Re-MX_data_24to28.07.2024\Corrected_27.07.2024\Corrected_12_Hours_sample\0001SET\layerstack\indices2"        # Root folder for indices
process_layerstacked_images(input_folder, output_root)

Processing F:\Re-MX_data_24to28.07.2024\Corrected_27.07.2024\Corrected_12_Hours_sample\0001SET\layerstack\IMG_0001_layerstacked.tif...
ARI saved to F:\Re-MX_data_24to28.07.2024\Corrected_27.07.2024\Corrected_12_Hours_sample\0001SET\layerstack\indices2\IMG_0001_layerstacked\ari.tif
CIRE saved to F:\Re-MX_data_24to28.07.2024\Corrected_27.07.2024\Corrected_12_Hours_sample\0001SET\layerstack\indices2\IMG_0001_layerstacked\cire.tif
CVI saved to F:\Re-MX_data_24to28.07.2024\Corrected_27.07.2024\Corrected_12_Hours_sample\0001SET\layerstack\indices2\IMG_0001_layerstacked\cvi.tif
DVI saved to F:\Re-MX_data_24to28.07.2024\Corrected_27.07.2024\Corrected_12_Hours_sample\0001SET\layerstack\indices2\IMG_0001_layerstacked\dvi.tif
EVI saved to F:\Re-MX_data_24to28.07.2024\Corrected_27.07.2024\Corrected_12_Hours_sample\0001SET\layerstack\indices2\IMG_0001_layerstacked\evi.tif
EVI2 saved to F:\Re-MX_data_24to28.07.2024\Corrected_27.07.2024\Corrected_12_Hours_sample\0001SET\layerstack\indices2\IMG_0001_l